# HRU window randomization — post-experiment analysis

Reads the CSV outputs of `src/run_window_randomization.py` and performs:

1. **Statistical testing** — per relative-position bin, HRU vs control (Mann–Whitney U);
   if no control group is present, HRU Δ vs 0 (Wilcoxon signed-rank). BH/FDR corrected.
2. **HRU × position heatmap** — per-transcript Δ profile across the sequence.
3. **Publication-quality figures** — Times New Roman, 300 dpi, PNG + PDF.

`delta = baseline_score − randomized_score`; larger Δ ⇒ randomizing that region lowers
the relation score ⇒ that region is more important for the predicted 5'–3' UTR relation.

> Run the heavy experiment on a GPU machine first, then point `RESULT_DIR` below at its
> output directory. This notebook is lightweight and runs anywhere.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, wilcoxon

# --- publication style (matches the other notebooks in this repo) ---
mpl.rcParams["figure.dpi"] = 300
mpl.rcParams["savefig.dpi"] = 300
mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["pdf.fonttype"] = 42   # keep text editable in vector output
mpl.rcParams["ps.fonttype"] = 42
sns.set_theme(style="ticks", font="Times New Roman", font_scale=1.1)

try:
    from statannotations.Annotator import Annotator
    HAS_STATANNOT = True
except Exception as e:  # noqa: BLE001
    HAS_STATANNOT = False
    print(f"[warn] statannotations unavailable ({e}); boxplots will omit star annotations")

try:
    from statsmodels.stats.multitest import multipletests
    HAS_SM = True
except Exception:  # noqa: BLE001
    HAS_SM = False
    print("[warn] statsmodels unavailable; FDR correction skipped")

In [ ]:
# --- config: point this at the experiment output directory (default: w20 main run) ---
RESULT_DIR = "../results/window_randomization_human_w20"
N_BINS = 10  # relative-position bins (should match what you want to visualize)

ANALYSIS_DIR = os.path.join(RESULT_DIR, "analysis")
os.makedirs(ANALYSIS_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(RESULT_DIR, "window_randomization_results.csv"))
groups = pd.read_csv(os.path.join(RESULT_DIR, "analyzed_groups.csv"))
print(f"{len(df)} window rows | groups: {df['group'].value_counts().to_dict()}")
print(f"regions: {sorted(df['region'].unique())} | modes: {sorted(df['mode'].unique())}")
df.head()

## Per-transcript binning

To avoid pseudo-replication (a transcript contributes several overlapping windows to one
bin), we first average Δ **within each transcript × position bin**, then run statistics
across transcripts.

In [ ]:
def bin_and_aggregate(df, n_bins):
    bins = np.linspace(0, 1, n_bins + 1)
    centers = (bins[:-1] + bins[1:]) / 2
    d = df.copy()
    d["pos_bin"] = pd.cut(d["rel_center"], bins=bins, labels=False, include_lowest=True)
    binned = (
        d.groupby(["ENST_ID", "GENE", "group", "region", "mode", "pos_bin"])["delta"]
        .mean()
        .reset_index()
    )
    binned["rel_pos"] = binned["pos_bin"].map(lambda b: centers[int(b)])
    return binned, centers


binned, CENTERS = bin_and_aggregate(df, N_BINS)
binned.to_csv(os.path.join(ANALYSIS_DIR, "binned_per_transcript.csv"), index=False)
REGIONS = sorted(df["region"].unique())
MODES = sorted(df["mode"].unique())
HAS_CONTROL = (binned["group"] == "control").any()
print(f"HAS_CONTROL = {HAS_CONTROL}")
binned.head()

## 1. Statistical testing

For every (region, mode, position-bin):
- if a control group exists → **Mann–Whitney U** comparing HRU vs control Δ,
- otherwise → **Wilcoxon signed-rank** testing whether HRU Δ differs from 0.

p-values are BH/FDR-corrected across all bins (per region×mode).

In [ ]:
def per_bin_stats(binned, region, mode, n_bins, has_control):
    sub = binned[(binned["region"] == region) & (binned["mode"] == mode)]
    rows = []
    for b in range(n_bins):
        bb = sub[sub["pos_bin"] == b]
        hru = bb[bb["group"] == "HRU"]["delta"].values
        ctrl = bb[bb["group"] == "control"]["delta"].values
        stat, p, test = np.nan, np.nan, "NA"
        if has_control and len(hru) > 0 and len(ctrl) > 0:
            stat, p = mannwhitneyu(hru, ctrl, alternative="two-sided")
            test = "MWU(HRU vs control)"
        elif len(hru) > 1 and np.any(hru != 0):
            try:
                stat, p = wilcoxon(hru)
                test = "Wilcoxon(HRU vs 0)"
            except ValueError:
                pass
        rows.append(
            {
                "region": region, "mode": mode, "pos_bin": b,
                "rel_pos": CENTERS[b],
                "hru_n": len(hru), "ctrl_n": len(ctrl),
                "hru_mean_delta": float(np.mean(hru)) if len(hru) else np.nan,
                "ctrl_mean_delta": float(np.mean(ctrl)) if len(ctrl) else np.nan,
                "stat": stat, "p": p, "test": test,
            }
        )
    res = pd.DataFrame(rows)
    valid = res["p"].notna()
    if HAS_SM and valid.sum() > 0:
        res.loc[valid, "p_fdr"] = multipletests(res.loc[valid, "p"], method="fdr_bh")[1]
    else:
        res["p_fdr"] = res["p"]
    return res


stats_tables = [
    per_bin_stats(binned, r, m, N_BINS, HAS_CONTROL) for r in REGIONS for m in MODES
]
stats_df = pd.concat(stats_tables, ignore_index=True)
stats_df.to_csv(os.path.join(ANALYSIS_DIR, "per_bin_stats.csv"), index=False)
stats_df

In [ ]:
def star(p):
    if pd.isna(p):
        return "ns"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    return "ns"


def boxplot_bins(binned, region, mode):
    sub = binned[(binned["region"] == region) & (binned["mode"] == mode)].copy()
    sub["pos_bin"] = sub["pos_bin"].astype(int)
    fig, ax = plt.subplots(figsize=(7, 4))
    if HAS_CONTROL:
        params = dict(data=sub, x="pos_bin", y="delta", hue="group",
                      order=sorted(sub["pos_bin"].unique()),
                      hue_order=["HRU", "control"], showfliers=False)
        sns.boxplot(ax=ax, **params)
        if HAS_STATANNOT:
            pairs = [((b, "HRU"), (b, "control")) for b in sorted(sub["pos_bin"].unique())]
            annot = Annotator(ax, pairs, **params)
            annot.configure(test="Mann-Whitney", text_format="star", loc="inside", verbose=0)
            annot.apply_and_annotate()
    else:
        params = dict(data=sub, x="pos_bin", y="delta",
                      order=sorted(sub["pos_bin"].unique()), showfliers=False,
                      color="#c0392b")
        sns.boxplot(ax=ax, **params)
        # annotate Wilcoxon-vs-0 stars above each bin
        st = stats_df[(stats_df["region"] == region) & (stats_df["mode"] == mode)]
        ymax = sub["delta"].quantile(0.98)
        for b in sorted(sub["pos_bin"].unique()):
            p = st[st["pos_bin"] == b]["p_fdr"].values
            ax.text(b, ymax, star(p[0] if len(p) else np.nan), ha="center", va="bottom", fontsize=10)
    ax.axhline(0, color="grey", lw=0.8, ls="--")
    ax.set_xlabel(f"Relative position bin in {region}  (0 = 5' end → {N_BINS-1} = 3' end)")
    ax.set_ylabel("Δ relation score")
    ax.set_title(f"{region}  ({mode})")
    sns.despine()
    fig.tight_layout()
    fig.savefig(os.path.join(ANALYSIS_DIR, f"box_delta_bins_{region}_{mode}.png"))
    fig.savefig(os.path.join(ANALYSIS_DIR, f"box_delta_bins_{region}_{mode}.pdf"))
    plt.show()


for r in REGIONS:
    for m in MODES:
        boxplot_bins(binned, r, m)

### Overall HRU vs control (most-sensitive window per transcript)

Compares the single largest Δ per transcript — does randomizing the *best* window hurt
HRUs more than control pairs?

In [ ]:
maxw = pd.read_csv(os.path.join(RESULT_DIR, "max_window_per_transcript.csv"))
for m in MODES:
    fig, ax = plt.subplots(figsize=(5, 4))
    sub = maxw[maxw["mode"] == m]
    if HAS_CONTROL:
        params = dict(data=sub, x="region", y="delta", hue="group",
                      hue_order=["HRU", "control"], showfliers=False)
        sns.boxplot(ax=ax, **params)
        if HAS_STATANNOT:
            pairs = [((r, "HRU"), (r, "control")) for r in sorted(sub["region"].unique())]
            annot = Annotator(ax, pairs, **params)
            annot.configure(test="Mann-Whitney", text_format="star", loc="inside", verbose=0)
            annot.apply_and_annotate()
        # also print the numeric p-values
        for r in sorted(sub["region"].unique()):
            h = sub[(sub.region == r) & (sub.group == "HRU")]["delta"]
            c = sub[(sub.region == r) & (sub.group == "control")]["delta"]
            if len(h) and len(c):
                print(f"[{m}] {r}: MWU p = {mannwhitneyu(h, c, alternative='two-sided')[1]:.2e}")
    else:
        sns.boxplot(ax=ax, data=sub, x="region", y="delta", showfliers=False, color="#c0392b")
    ax.set_ylabel("max Δ relation score (per transcript)")
    ax.set_title(f"Most-sensitive window ({m})")
    sns.despine()
    fig.tight_layout()
    fig.savefig(os.path.join(ANALYSIS_DIR, f"box_max_delta_{m}.png"))
    fig.savefig(os.path.join(ANALYSIS_DIR, f"box_max_delta_{m}.pdf"))
    plt.show()

## 2. HRU × position heatmap

Rows = HRUs, columns = relative-position bins, colour = mean Δ. Reveals whether
HRUs depend on a localized region (a hot column) or are broadly distributed, and lets
you cluster HRUs by their sensitivity profile. Missing bins (no window centered there)
are treated as 0.

In [ ]:
def hru_heatmap(binned, region, mode, n_bins, normalize=False, cluster_rows=True):
    sub = binned[(binned["region"] == region) & (binned["mode"] == mode) & (binned["group"] == "HRU")]
    if len(sub) == 0:
        print(f"no HRU rows for {region}/{mode}")
        return
    piv = sub.pivot_table(index="ENST_ID", columns="pos_bin", values="delta", aggfunc="mean")
    piv = piv.reindex(columns=range(n_bins)).fillna(0.0)
    title = f"HRU Δ profile: {region} ({mode})"
    if normalize:  # per-row min-max -> compare shape regardless of magnitude
        rng = (piv.max(axis=1) - piv.min(axis=1)).replace(0, 1)
        piv = piv.sub(piv.min(axis=1), axis=0).div(rng, axis=0)
        title += " [row-normalized]"
    g = sns.clustermap(
        piv, cmap="Reds", col_cluster=False, row_cluster=cluster_rows,
        figsize=(5, 9), yticklabels=False,
        cbar_kws={"label": "Δ relation score"},
    )
    g.ax_heatmap.set_xlabel(f"position bin (0 = 5' → {n_bins-1} = 3')")
    g.ax_heatmap.set_ylabel(f"HRUs (n={piv.shape[0]})")
    g.figure.suptitle(title, y=1.02)
    g.savefig(os.path.join(ANALYSIS_DIR, f"heatmap_hru_{region}_{mode}.png"))
    g.savefig(os.path.join(ANALYSIS_DIR, f"heatmap_hru_{region}_{mode}.pdf"))
    plt.show()


for r in REGIONS:
    for m in MODES:
        hru_heatmap(binned, r, m, N_BINS, normalize=False)

## 3. Publication-quality summary figure

Mean Δ vs relative position, one panel per region, lines per group (and mode), with SEM
bands. This is the headline interpretability figure for the manuscript.

In [ ]:
def sem(x):
    x = np.asarray(x, dtype=float)
    return x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0


palette = {"HRU": "#c0392b", "control": "#2c3e50"}
line_styles = {"uniform": "-", "shuffle": "--"}

fig, axes = plt.subplots(1, len(REGIONS), figsize=(5.2 * len(REGIONS), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, region in zip(axes, REGIONS):
    sub = binned[binned["region"] == region]
    for (group, mode), gdf in sub.groupby(["group", "mode"]):
        prof = gdf.groupby("rel_pos")["delta"].agg(["mean", sem]).reset_index()
        label = f"{group} ({mode})" if len(MODES) > 1 else group
        ax.plot(prof["rel_pos"], prof["mean"], line_styles.get(mode, "-"),
                marker="o", ms=4, color=palette.get(group, None), label=label)
        ax.fill_between(prof["rel_pos"], prof["mean"] - prof["sem"], prof["mean"] + prof["sem"],
                        color=palette.get(group, None), alpha=0.15)
    ax.axhline(0, color="grey", lw=0.8, ls=":")
    ax.set_xlabel(f"Relative position in {region}\n(0 = 5' end, 1 = 3' end)")
    ax.set_title(region)
    ax.legend(frameon=False, fontsize=9)
axes[0].set_ylabel("Δ relation score (baseline − randomized)")
sns.despine()
fig.tight_layout()
fig.savefig(os.path.join(ANALYSIS_DIR, "summary_delta_vs_position.png"))
fig.savefig(os.path.join(ANALYSIS_DIR, "summary_delta_vs_position.pdf"))
plt.show()
print(f"All analysis outputs saved under: {ANALYSIS_DIR}")

### Notes / interpretation

- **Δ > 0** at a position ⇒ randomizing it lowers the relation score ⇒ that region drives
  the predicted relationship. E.g. a peak at the 3' end of the 5'UTR (near the CDS) would
  align with the manuscript's MFE finding (low MFE / stronger structure at the 5'UTR tail).
- **uniform − shuffle** ≈ contribution of base **composition**; **shuffle** Δ ≈ contribution
  of **order / secondary structure** (composition held fixed).
- **HRU vs control**: significant, localized HRU peaks (vs flat control) answer the
  reviewer's "what distinguishes HRUs / what do we learn beyond pair-matching".
- See `per_bin_stats.csv` for exact (FDR-corrected) p-values.